# NumPy Fundamentals
**Ringkasan praktik 9 topik dasar NumPy** — merujuk pada dokumentasi resmi
[NumPy fundamentals](https://numpy.org/doc/2.5/user/basics.html) (NumPy v2.5).

Topik yang dibahas:
1. Array creation — pembuatan array
2. Indexing on ndarrays — pengindeksan
3. I/O with NumPy — simpan & baca data
4. Data types — tipe data
5. Broadcasting — penyiaran operasi
6. Copies and views — salinan vs tampilan
7. Working with Arrays of Strings and Bytes
8. Structured arrays — array terstruktur
9. Universal functions (ufunc) basics

> Notebook ini dieksekusi dengan NumPy versi di bawah ini, sehingga semua
> output adalah hasil nyata, bukan karangan.

In [1]:
import numpy as np
print("NumPy", np.__version__)

NumPy 2.5.3


## 1. Array Creation — Pembuatan Array

Inti NumPy adalah `ndarray`: larik n-dimensi yang homogen (semua elemen
bertipe sama) dan tersimpan rapat di memori. Cara paling umum membuatnya:

In [2]:
# Dari list Python
a = np.array([1, 2, 3])
print(a, a.shape, a.ndim)

# 2 dimensi
b = np.array([[1, 2, 3], [4, 5, 6]])
print(b)
print("shape:", b.shape)   # (baris, kolom)

[1 2 3] (3,) 1
[[1 2 3]
 [4 5 6]]
shape: (2, 3)


In [3]:
# Array khusus: nol, satu, kosong, identitas
print(np.zeros((2, 3)))
print(np.ones((2, 2), dtype=int))
print(np.eye(3))            # matriks identitas

[[0. 0. 0.]
 [0. 0. 0.]]
[[1 1]
 [1 1]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [4]:
# Barisan bilangan: arange & linspace
print(np.arange(0, 10, 2))          # start, stop, step
print(np.linspace(0, 1, 5))         # 5 titik merata 0..1

# Mengubah bentuk tanpa mengubah data: reshape
x = np.arange(12).reshape(3, 4)
print(x)

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


Fungsi khusus 2D: `diag` (diagonal) dan `vander` (matriks Vandermonde).

In [5]:
print(np.diag([1, 2, 3]))
print(np.diag(np.array([[1, 2], [3, 4]])))  # ekstrak diagonal
print(np.vander([1, 2, 3], 3))

[[1 0 0]
 [0 2 0]
 [0 0 3]]
[1 4]
[[1 1 1]
 [4 2 1]
 [9 3 1]]


Bilangan acak yang *reproducible* memakai `default_rng` dengan seed; serta penggabungan array dengan `vstack`, `hstack`, `block`.

In [6]:
from numpy.random import default_rng
rng = default_rng(42)
print(rng.random((2, 3)))   # selalu sama tiap dijalankan

a = np.ones((2, 2)); b = np.zeros((2, 2))
print(np.hstack([a, b]))
print(np.vstack([a, b]))

[[0.77395605 0.43887844 0.85859792]
 [0.69736803 0.09417735 0.97562235]]
[[1. 1. 0. 0.]
 [1. 1. 0. 0.]]
[[1. 1.]
 [1. 1.]
 [0. 0.]
 [0. 0.]]


Operasi antar dtype berbeda mengikuti *type promotion*: NumPy memilih tipe yang mampu menampung semua nilai.

In [7]:
a = np.array([2, 3, 4], dtype=np.uint32)
print((a - np.array([5, 6, 7], dtype=np.uint32)).dtype)  # tetap uint32 (meluap!)
print((a - np.array([5, 6, 7], dtype=np.int32)).dtype)  # dipromosikan ke int64

uint32
int64


## 2. Indexing on ndarrays — Pengindeksan

Indeks dimulai dari 0. Selain indeks tunggal, NumPy mendukung *slicing*,
*boolean indexing*, dan *fancy indexing*.

In [8]:
a = np.arange(10)
print(a[0], a[-1])        # elemen pertama & terakhir
print(a[2:6])             # slice: indeks 2..5
print(a[::2])             # tiap 2 langkah

0 9
[2 3 4 5]
[0 2 4 6 8]


In [9]:
m = np.arange(12).reshape(3, 4)
print(m)
print("baris 1:", m[1])
print("kolom 2:", m[:, 2])
print("blok 2x2:", m[0:2, 1:3], sep="\n")

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
baris 1: [4 5 6 7]
kolom 2: [ 2  6 10]
blok 2x2:
[[1 2]
 [5 6]]


In [10]:
# Boolean indexing: pilih elemen yang memenuhi syarat
m = np.arange(12).reshape(3, 4)
print(m[m > 6])                 # semua elemen > 6
m[m % 2 == 0] = -1              # tulis ke posisi genap
print(m)

# Fancy indexing: indeks berupa array
a = np.arange(10)
print(a[[1, 3, 5]])             # ambil indeks 1,3,5

[ 7  8  9 10 11]
[[-1  1 -1  3]
 [-1  5 -1  7]
 [-1  9 -1 11]]
[1 3 5]


`Ellipsis` (`...`) mewakili semua dimensi yang tersisa, dan `newaxis` menyisipkan dimensi baru berukuran 1 — berguna untuk broadcasting.

In [11]:
x = np.arange(24).reshape(2, 3, 4)
print(x[..., 0].shape)          # sama dengan x[:, :, 0]
print(x[:, np.newaxis, :, :].shape)  # (2, 1, 3, 4)

# tabel penjumlahan via newaxis + broadcasting
v = np.arange(5)
print(v[:, np.newaxis] + v[np.newaxis, :])

(2, 3)
(2, 1, 3, 4)
[[0 1 2 3 4]
 [1 2 3 4 5]
 [2 3 4 5 6]
 [3 4 5 6 7]
 [4 5 6 7 8]]


*Integer array indexing* multidimensi: pasangan array indeks dibroadcast bersama, hasilnya mengikuti bentuk array indeks.

In [12]:
y = np.arange(35).reshape(5, 7)
print(y[np.array([0, 2, 4]), np.array([0, 1, 2])])  # y[0,0], y[2,1], y[4,2]
# np.ix_: produk silang baris x kolom
print(y[np.ix_([0, 2], [1, 3])])

[ 0 15 30]
[[ 1  3]
 [15 17]]


## 3. I/O with NumPy — Simpan & Baca Data

NumPy bisa menyimpan array ke file teks (terbaca manusia) atau biner
(lebih cepat & ringkas).

In [13]:
a = np.arange(12).reshape(3, 4)

# Teks: savetxt / loadtxt
np.savetxt("/tmp/data.txt", a, fmt="%d", delimiter=",")
print(open("/tmp/data.txt").read())
b = np.loadtxt("/tmp/data.txt", delimiter=",", dtype=int)
print("terbaca kembali:\n", b)

0,1,2,3
4,5,6,7
8,9,10,11

terbaca kembali:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


In [14]:
# Biner: save / load (.npy) dan savez (banyak array, .npz)
np.save("/tmp/array.npy", a)
np.savez("/tmp/banyak.npz", pertama=a, kedua=a * 2)

print(np.load("/tmp/array.npy"))
d = np.load("/tmp/banyak.npz")
print(d.files)
print(d["kedua"])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
['pertama', 'kedua']
[[ 0  2  4  6]
 [ 8 10 12 14]
 [16 18 20 22]]


`genfromtxt` lebih fleksibel dari `loadtxt`: menangani header dan data hilang (missing values).

In [15]:
open("/tmp/data2.csv", "w").write("x,y\n0,0\n1,\n2,4\n")
print(np.genfromtxt("/tmp/data2.csv", delimiter=",", names=True))

[(0.,  0.) (1., nan) (2.,  4.)]


## 4. Data Types — Tipe Data

Setiap `ndarray` punya satu `dtype`. NumPy menyediakan tipe numerik
(`int8`…`int64`, `float16/32/64`, `complex`), boolean, string, dan lain-lain.
Pemilihan dtype memengaruhi **ketelitian** dan **pemakaian memori**.

In [16]:
a = np.array([1, 2, 3])
print(a.dtype)  # default = np.int_, tergantung platform (umumnya int64 di Linux/Mac, int32 di Windows)

b = np.array([1, 2, 3], dtype=np.float32)
print(b.dtype, b.nbytes, "byte")     # 3 * 4 byte

c = np.array([1.7, 2.3])
print(c.astype(int))                 # konversi eksplisit

int64
float32 12 byte
[1 2]


In [17]:
# Hati-hati overflow pada tipe kecil!
d = np.array([200], dtype=np.uint8)
print(d + 100)        # uint8 maks 255 -> meluap!

# itemsize & nbytes untuk ukur memori
e = np.zeros((1000, 1000), dtype=np.float64)
print(f"{e.nbytes / 1e6:.1f} MB untuk 1000x1000 float64")

[44]
8.0 MB untuk 1000x1000 float64


`iinfo` / `finfo` menampilkan batas dan presisi suatu tipe.

In [18]:
print(np.iinfo(np.int32))
print(np.finfo(np.float32).eps)   # epsilon mesin

Machine parameters for int32
---------------------------------------------------------------
min = -2147483648
max = 2147483647
---------------------------------------------------------------

1.1920929e-07


## 5. Broadcasting — Penyiaran Operasi

*Broadcasting* memungkinkan operasi antara array berukuran berbeda **tanpa
menyalin data**: dimensi yang berukuran 1 (atau hilang) "disiarkan"
mengikuti dimensi yang lebih besar. Syaratnya: ukuran tiap dimensi harus
sama, atau salah satunya 1.

In [19]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])          # shape (2, 3)
b = np.array([10, 20, 30])         # shape (3,) -> disiarkan ke (2, 3)
print(a + b)

[[11 22 33]
 [14 25 36]]


In [20]:
# Contoh citra: naikkan brightness tiap channel RGB berbeda
citra = np.zeros((2, 2, 3))        # 2x2 piksel, 3 channel
koreksi = np.array([10, 20, 30])   # per channel
print((citra + koreksi).shape)     # (2,2,3) tetap

# Skalar juga di-broadcast
print(np.arange(4) * 2.5)

(2, 2, 3)
[0.  2.5 5.  7.5]


In [21]:
# Yang GAGAL: (3,) dengan (4,) -> tidak kompatibel
try:
    np.ones(3) + np.ones(4)
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (3,) (4,) 


## 6. Copies and Views — Salinan vs Tampilan

Ini konsep paling sering menjebak pemula:

- **View**: "jendela" ke data yang sama — mengubah view mengubah aslinya.
  Slice dasar (`a[2:5]`) menghasilkan view.
- **Copy**: data diduplikasi — aman diubah. `a.copy()` atau fancy indexing
  menghasilkan copy.

In [22]:
a = np.arange(10)
v = a[2:5]          # VIEW
v[0] = 99
print("a berubah juga:", a)   # a ikut berubah!

b = np.arange(10)
c = b[2:5].copy()   # COPY
c[0] = -1
print("b tetap:", b)          # b aman

a berubah juga: [ 0  1 99  3  4  5  6  7  8  9]
b tetap: [0 1 2 3 4 5 6 7 8 9]


In [23]:
# reshape juga view bila memungkinkan
m = np.arange(12).reshape(3, 4)
print("base is m:", m.reshape(12).base is m)

# Cek cepat: atribut .base
x = np.arange(5)
print("slice .base:", x[1:3].base is x)   # True -> view
print("copy  .base:", x[1:3].copy().base) # None -> data sendiri

base is m: False
slice .base: True
copy  .base: None


`ravel` mengembalikan view bila memungkinkan, sedangkan `flatten` selalu mengembalikan copy.

In [24]:
m = np.arange(12).reshape(3, 4)
print("ravel is view:", m.ravel().base is m)
print("flatten is copy:", m.flatten().base is None)

ravel is view: False
flatten is copy: True


## 7. Working with Arrays of Strings and Bytes

NumPy punya dtype string Unicode `<U` (panjang tetap) dan bytes `|S`.
Panjangnya ditentukan oleh string terpanjang saat pembuatan — string yang
lebih panjang akan **dipotong**!

In [25]:
nama = np.array(["budi", "ani", "candra"])
print(nama.dtype)          # <U6 (6 karakter)

# Operasi vektorisasi pada string
print(np.char.upper(nama))
print(np.char.add(nama, "@mail.com"))

<U6
['BUDI' 'ANI' 'CANDRA']
['budi@mail.com' 'ani@mail.com' 'candra@mail.com']


In [26]:
# Awas: dtype panjang-tetap memotong!
s = np.array(["ab", "cdef"])
print(s.dtype)             # <U4
s[0] = "abcdefgh"
print(s)                   # terpotong jadi 'abcd'

<U4
['abcd' 'cdef']


## 8. Structured Arrays — Array Terstruktur

Array terstruktur menyimpan **baris-baris record** dengan kolom bertipe
berbeda — mirip satu tabel mini: tiap elemen punya *field* bernama.

In [27]:
dt = np.dtype([("nama", "U10"), ("nim", "i4"), ("ipk", "f4")])
mhs = np.array([("budi", 101, 3.5), ("ani", 102, 3.8)], dtype=dt)
print(mhs)
print("nama:", mhs["nama"])
print("ipk :", mhs["ipk"])

[('budi', 101, 3.5) ('ani', 102, 3.8)]
nama: ['budi' 'ani']
ipk : [3.5 3.8]


In [28]:
# Filter baris berdasar field
print(mhs[mhs["ipk"] > 3.6])

[('ani', 102, 3.8)]


Dtype terstruktur dapat ditulis dalam beberapa notasi: list of tuples, string ringkas, atau dict (paling fleksibel, bisa atur offset byte). Field juga dapat berupa subarray.

In [29]:
print(np.dtype("i8, f4, S3"))                              # string ringkas
print(np.dtype({"names": ["x", "y"], "formats": ["i4", "f4"]}))  # dict
dt2 = np.dtype([("pos", "f4", (2,)), ("label", "U5")])      # subarray (2,)
r = np.array([([1.0, 2.0], "titik"), ([3.0, 4.0], "lain")], dtype=dt2)
print(r["pos"])

[('f0', '<i8'), ('f1', '<f4'), ('f2', 'S3')]
[('x', '<i4'), ('y', '<f4')]
[[1. 2.]
 [3. 4.]]


## 9. Universal Functions (ufunc) Basics

*Ufunc* adalah fungsi yang bekerja **elemen-per-elemen** pada array,
diimplementasikan di C sehingga jauh lebih cepat daripada loop Python.
Contoh: `np.add`, `np.sqrt`, `np.exp`, operator `+ - * /` pun memanggil ufunc.

In [30]:
a = np.array([1, 4, 9, 16])
print(np.sqrt(a))          # ufunc uner
print(np.add(a, 10))       # ufunc biner == a + 10
print(np.maximum(a, 10))   # perbandingan elemen-per-elemen

[1. 2. 3. 4.]
[11 14 19 26]
[10 10 10 16]


In [31]:
# Kecepatan: ufunc vs loop Python
import time
besar = np.arange(1_000_000, dtype=float)
t0 = time.perf_counter(); r1 = np.sqrt(besar); t1 = time.perf_counter()
t2 = time.perf_counter()
r2 = np.array([x ** 0.5 for x in besar]); t3 = time.perf_counter()
print(f"ufunc: {t1-t0:.4f} dtk | loop: {t3-t2:.4f} dtk")

ufunc: 0.0095 dtk | loop: 0.3412 dtk


In [32]:
# Atribut ufunc yang berguna
print("nin/nout:", np.add.nin, np.add.nout)
# reduce: agregasi sepanjang array
print("jumlah:", np.add.reduce([1, 2, 3, 4]))
print("kumulatif:", np.add.accumulate([1, 2, 3, 4]))

nin/nout: 2 1
jumlah: 10
kumulatif: [ 1  3  6 10]


Metode `outer`: produk luar — menerapkan ufunc ke semua pasangan elemen dua array (memanfaatkan broadcasting).

In [33]:
print(np.multiply.outer([1, 2, 3], [10, 20]))
print(np.add.outer(np.arange(3), np.arange(3)))

[[10 20]
 [20 40]
 [30 60]]
[[0 1 2]
 [1 2 3]
 [2 3 4]]


Metode `at`: operasi in-place tanpa buffer — indeks yang disebut berulang akan diproses berulang (berbeda dengan indexing biasa).

In [34]:
a = np.zeros(5, dtype=int)
np.add.at(a, [1, 1, 3], 1)   # indeks 1 ditambah dua kali
print(a)

[0 2 0 1 0]


### Ufunc biasa vs *generalized ufunc*

Ufunc biasa intinya bekerja pada **skalar**: satu elemen diproses dalam satu
waktu (misalnya `np.add` menjumlahkan elemen-per-elemen). Ada juga
*generalized ufunc* (gufunc) yang inti operasinya berupa **sub-array**
(vektor/matriks), misalnya `np.matmul` untuk perkalian matriks.

In [35]:
a = np.arange(6).reshape(3, 2)
print("a =\n", a)
print("np.add (skalar, elemen-per-elemen) =\n", np.add(a, a))
print("np.matmul (matriks 3x2 @ 2x3) =\n", np.matmul(a, a.T))

a =
 [[0 1]
 [2 3]
 [4 5]]
np.add (skalar, elemen-per-elemen) =
 [[ 0  2]
 [ 4  6]
 [ 8 10]]
np.matmul (matriks 3x2 @ 2x3) =
 [[ 1  3  5]
 [ 3 13 23]
 [ 5 23 41]]


### Membuat ufunc sendiri dengan `np.frompyfunc`

Fungsi Python biasa bisa "divektorkan" menjadi ufunc lewat
`np.frompyfunc(fungsi, nin, nout)`. Hasilnya selalu ber-`dtype=object`,
jadi cocok untuk operasi yang belum tersedia sebagai ufunc bawaan
(misalnya operasi string).

In [36]:
def sapa(nama):
    return f"Halo, {nama}!"

sapa_vec = np.frompyfunc(sapa, 1, 1)   # 1 input, 1 output
hasil = sapa_vec(np.array(["Ani", "Budi", "Citra"]))
print(hasil)
print("dtype hasil:", hasil.dtype)

['Halo, Ani!' 'Halo, Budi!' 'Halo, Citra!']
dtype hasil: object


### `reduceat`: reduksi per segmen

Metode `reduceat` mereduksi **potongan-potongan** array sekaligus berdasarkan
daftar indeks awal tiap segmen. `np.add.reduceat(x, [0, 3, 7])` menjumlahkan
`x[0:3]`, `x[3:7]`, dan `x[7:]` dalam satu panggilan.

In [37]:
x = np.arange(10)
print(np.add.reduceat(x, [0, 3, 7]))   # [0+1+2, 3+4+5+6, 7+8+9]

[ 3 18 24]


### Argumen `axis`, `dtype`, dan `out` pada metode reduksi

Metode seperti `reduce`/`accumulate` menerima tiga argumen kunci:

- `axis`: sumbu yang direduksi — bisa berupa *tuple* untuk mereduksi
  beberapa sumbu sekaligus;
- `dtype`: tipe data yang dipakai **selama komputasi** (mencegah overflow
  atau mengatur presisi hasil);
- `out`: array penampung hasil, agar tidak mengalokasi array baru.

In [38]:
x = np.arange(9).reshape(3, 3)
print("jumlah per baris (axis=1):\n", np.add.reduce(x, 1))
print("jumlah semua elemen (axis=(0, 1)):", np.add.reduce(x, (0, 1)))
print("perkalian per baris, dihitung sbg float64:\n",
      np.multiply.reduce(x, dtype=np.float64))

wadah = np.zeros(3, dtype=np.int64)
np.add.reduce(x, axis=1, out=wadah)   # tulis langsung ke 'wadah'
print("hasil via out=:", wadah)

jumlah per baris (axis=1):
 [ 3 12 21]
jumlah semua elemen (axis=(0, 1)): 36
perkalian per baris, dihitung sbg float64:
 [ 0. 28. 80.]
hasil via out=: [ 3 12 21]


### Aturan *type casting* pada ufunc

Di balik layar, tiap ufunc menyimpan daftar *inner loop* untuk kombinasi
tipe tertentu (intip via `np.add.types`). Bila tipe input tidak punya loop
yang cocok, NumPy mencari loop yang bisa dicapai lewat *casting* yang
**aman** — cek dengan `np.can_cast(dari, ke)`. Praktisnya:
`int32 -> float64` aman, sebaliknya tidak.

Aturan khusus untuk **skalar Python**: skalar tidak "menaikkan" tipe array
selama masih sejenis (array `int32` + skalar `5` tetap `int32`); ia hanya
menaikkan tipe bila jenisnya berbeda (array `int32` + skalar `1.5`
menjadi `float64`).

In [39]:
print("int32 -> float64 aman?", np.can_cast(np.int32, np.float64))
print("float64 -> int32 aman?", np.can_cast(np.float64, np.int32))
print("contoh inner loop np.add:", np.add.types[:4], "...")
print("int32 + 5   ->", (np.arange(3, dtype=np.int32) + 5).dtype)
print("int32 + 1.5 ->", (np.arange(3, dtype=np.int32) + 1.5).dtype)

int32 -> float64 aman? True
float64 -> int32 aman? False
contoh inner loop np.add: ['??->?', 'bb->b', 'BB->B', 'hh->h'] ...
int32 + 5   -> int32
int32 + 1.5 -> float64


### Tipe hasil: *array scalar* dan argumen `out`

Hasil ufunc atas `ndarray` adalah `ndarray` — **kecuali** hasilnya
berdimensi nol (misalnya `reduce` tanpa `axis`), yang dikembalikan sebagai
*array scalar* (misalnya `numpy.int64`). Memberi argumen `out=` memaksa
hasil ditulis ke array yang kita sediakan.

In [40]:
r = np.add.reduce([1, 2, 3])
print(type(r), "->", r)                 # skalar numpy, bukan ndarray
print(type(np.add([1, 2], [3, 4])))     # ini tetap ndarray

tampung = np.empty(2, dtype=np.int64)
print(np.add([1, 2], [3, 4], out=tampung))  # hasil ditulis ke 'tampung'

<class 'numpy.int64'> -> 6
<class 'numpy.ndarray'>
[4 6]


### Penanganan error floating-point & buffer internal

Operasi seperti pembagian dengan nol memicu status *floating-point*
(peringatan `RuntimeWarning`). Pengaturan default bisa dilihat dengan
`np.geterr()` dan diubah global via `np.seterr()`; untuk perubahan
**sementara dan aman**, gunakan *context manager* `np.errstate`.

Di balik layar, ufunc memakai buffer internal (ukurannya via
`np.getbufsize()`) untuk data yang perlu dirapikan/dikonversi sebelum
dihitung.

In [41]:
print("pengaturan error default:", np.geterr())
with np.errstate(divide="ignore", invalid="ignore"):
    print(np.array([1.0, 0.0]) / np.array([0.0, 0.0]))   # tanpa warning
print("ukuran buffer internal (elemen):", np.getbufsize())

pengaturan error default: {'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}
[inf nan]
ukuran buffer internal (elemen): 8192


## Penutup

Sembilan fondasi NumPy di atas — pembuatan array, indexing, I/O, dtype,
broadcasting, copy/view, string, structured array, dan ufunc — adalah bekal
wajib sebelum masuk ke pengolahan citra digital (citra = `ndarray` 2D/3D!).

**Referensi:** dokumentasi resmi NumPy v2.5, *NumPy fundamentals* —
https://numpy.org/doc/2.5/user/basics.html